# FPL Squad Optimizer - Dashboard

This notebook uses the refactored `fpl_engine` modules to run the sequential transfer planning and optimization pipeline.

In [1]:
# Run this if you are in Google Colab to mount your drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import sys
    sys.path.append('/content/drive/My Drive/Hobby/FPL')
except ImportError:
    pass  # Not running in Colab

In [2]:
%load_ext autoreload
%autoreload 2
import asyncio
import pandas as pd
import nest_asyncio
nest_asyncio.apply()

from fpl_engine.config import (
    MY_FPL_ID, current_realizable_value_dict, get_season_params,
)
from fpl_engine.data import (
    get_current_players_df, fetch_raw_history_cache,
    get_team_df, get_pos_constraint_df,
    get_current_gameweek, get_max_finished_gameweek,
    get_fpl_gameweek_data, get_my_player_ids,
    get_dynamic_weights,
)
from fpl_engine.features import (
    compute_rolling_team_ratings, blend_team_ratings,
    get_fixture_players_stats_df, compute_global_z_distributions,
)
from fpl_engine.scoring import (
    _fit_bonus_multinomial, _fit_regression_params,
    _calculate_performance_indices, create_optimized_custom_score,
)
from fpl_engine.solver import plan_sequential_transfers

## 1. Configuration

In [3]:
bank_values = 1.1
current_free_transfer_avaliable = 1
default_horizon = 3

max_diff_weight = 0.13
max_upside_weight = 0.12

## 2. Fetch Data & Build Features

In [4]:
# 1. Fetch Base Data
current_gw = get_current_gameweek()
data_gameweek = get_max_finished_gameweek()
fpl_team_df = get_team_df()
players_df = get_current_players_df()

# 2. Season-Adaptive Params (interpolated for current GW)
params = get_season_params(current_gw)

print(f"Current Gameweek: {current_gw}")
print(f"Max Finished GW:  {data_gameweek}")

# 3. Fetch Match History (Async)
active_player_ids = players_df[players_df['minutes'] > 0]['id'].tolist()
raw_history_df = await fetch_raw_history_cache(active_player_ids, use_cache=True)

Current Gameweek: 36
Max Finished GW:  36
Cache raw_history_cache.parquet expired (older than 12 hours). Fetching fresh data...


Fetching Match History (Async):   0%|          | 0/532 [00:00<?, ?it/s]

RuntimeError: Timeout context manager should be used inside a task

In [ ]:
# 4. Compute Team Ratings & Blends
rolling_ratings_raw, latest_ratings_raw = compute_rolling_team_ratings(
    raw_history_df,
    ema_alpha    = params.get('rolling_ema_alpha', 0.15),
    min_fixtures = 3,
)

team_ratings_df, latest_team_ratings = blend_team_ratings(
    rolling_ratings_raw,
    latest_ratings_raw,
    fpl_team_df,
    league_avg_xG           = params.get('league_avg_xG',            1.45),
    league_avg_xGC          = params.get('league_avg_xGC',           1.45),
    blend_alpha             = params.get('blend_alpha',              0.75),
    min_fixtures_full_trust = params.get('min_fixtures_full_trust',  10),
)

# 5. Generate Fixture Specific Projections
global_dists = compute_global_z_distributions(team_ratings_df)

fixture_player_df = get_fixture_players_stats_df(
    params,
    raw_history_df,
    global_dists,
    team_ratings_df    = team_ratings_df,
    latest_team_ratings= latest_team_ratings,
)

## 3. Generate Scoring Projections

In [ ]:
# 1. Fit Models & Inject Regression Weights
reg_params  = _fit_regression_params(fixture_player_df)
bonus_model = _fit_bonus_multinomial(raw_history_df)
params.update(reg_params)

# 2. Calculate Performance Indices
fixture_player_df = _calculate_performance_indices(
    fixture_player_df,
    params,
    bonus_model=bonus_model
)

In [ ]:
# 3. Aggregate to per-GW projections (sum scoring, mean rates)
grouping_columns = [
    'gameweek', 'id_player', 'now_cost', 'selected_by_percent',
    'web_name', 'position', 'team_name'
]

sum_columns = [
    'Perf_IDX', 'ceiling_score', 'GOAL_INDEX', 'ASSIST_INDEX',
    'CLEAN_SHEET_INDEX', 'bonus_component', 'defcon_component',
    'minutes_IDX', 'actual_minutes'
]

mean_columns = [
    'recent_minutes_form', 'finishing_factor', 'protective_factor',
    'fixture_attack_multiplier', 'fixture_defence_multiplier',
    'fixture_calibrated_points',
    'start_per_gameplayed', 'consecutive_start_streak', 'hybrid_bps_abs', 'score_std',
]

# Build agg_dict: only include columns that actually exist
agg_dict = {col: 'sum' for col in sum_columns if col in fixture_player_df.columns}
agg_dict.update({col: 'mean' for col in mean_columns if col in fixture_player_df.columns})

# Only use grouping columns that exist
valid_grouping = [c for c in grouping_columns if c in fixture_player_df.columns]

gw_projection_df = fixture_player_df.groupby(valid_grouping).agg(agg_dict).reset_index()

print(f"Aggregated: {len(fixture_player_df)} fixture rows -> {len(gw_projection_df)} GW projection rows")

In [ ]:
# 4. Dynamic Weights & Custom Score
try:
    fpl_gameweek_data = get_fpl_gameweek_data(MY_FPL_ID)
    weights = get_dynamic_weights(fpl_gameweek_data, data_gameweek,
                                  max_diff_weight=max_diff_weight,
                                  max_upside_weight=max_upside_weight)
    diff_weight   = weights['diff_weight']
    upside_weight = weights['upside_weight']
    print(f"Mode: {weights['mode']}  |  Diff: {diff_weight:.4f}  |  Upside: {upside_weight:.4f}")
except Exception as e:
    print(f"Weight fetch failed: {e} -- using max weights")
    diff_weight   = max_diff_weight
    upside_weight = max_upside_weight

gw_projection_df = create_optimized_custom_score(
    df=gw_projection_df,
    differential_weight=diff_weight,
    upside_weight=upside_weight,
    # visualize=True
)

## 4. Projection Rankings

In [ ]:
projection_group = ['id_player', 'now_cost', 'selected_by_percent', 'web_name',
       'position', 'team_name', ]
avg_columns = list(set(list(agg_dict.keys()) + ['custom_score', 'ceiling_score', 'dynamic_upside']))
# Only keep columns that exist
avg_columns = [c for c in avg_columns if c in gw_projection_df.columns]

display_columns = [
    'id_player', 'now_cost', 'selected_by_percent', 'web_name', 'position',
    'team_name', 'minutes_IDX', 'Perf_IDX', 'ceiling_score',
    'GOAL_INDEX', 'ASSIST_INDEX', 'CLEAN_SHEET_INDEX', 'bonus_component',
    'defcon_component', 'finishing_factor', 'fixture_attack_multiplier',
    'protective_factor', 'fixture_defence_multiplier',
]
display_columns = [c for c in display_columns if c in gw_projection_df.columns]

In [ ]:
N = 15

# 1. Define the custom sorting order
pos_order = ['GKP', 'DEF', 'MID', 'FWD']

# 2. Filter and Process
top_n_df = gw_projection_df[gw_projection_df['gameweek'] >= current_gw+1].copy()
top_n_df = top_n_df.groupby(projection_group)[avg_columns].mean().reset_index()

# Convert position to a categorical type so it sorts correctly
top_n_df['position'] = pd.Categorical(top_n_df['position'], categories=pos_order, ordered=True)

# --- Safely combine columns without creating duplicates ---
final_cols = ['position', 'custom_score'] + display_columns
final_cols = list(dict.fromkeys(final_cols))
final_cols = [c for c in final_cols if c in top_n_df.columns]

# 3. Apply grouping and selection
result = (top_n_df
          .sort_values(['position', 'custom_score'], ascending=[True, False])
          .groupby('position', observed=True)
          .head(N)
          )[final_cols]

# 4. Print as a clean string
print(f"--- Top {N} Projections from GW{current_gw+1} + ---")
for pos in pos_order:
    pos_data = result[result['position'] == pos].round(3)
    print(f"\n== {pos} ==")
    if pos_data.empty:
        print("No data found.")
    else:
        print(pos_data.drop(columns=['position']).to_string(index=False))

## 5. Sequential Transfer Planner

In [ ]:
MY_CURRENT_TEAM_IDS = get_my_player_ids(MY_FPL_ID, current_gw)
locked_values = sum(current_realizable_value_dict.values()) + bank_values

print(f"My Net Transfer Value : {locked_values:.1f} M")
print(f"Bank : {bank_values:.1f} M")

plan_sequential_transfers(
    gw_projection_df = gw_projection_df,
    current_team_ids = MY_CURRENT_TEAM_IDS,
    start_gameweek = current_gw + 1,
    planning_horizon = default_horizon,
    initial_free_transfers = current_free_transfer_avaliable,
    current_realizable_value_dict = current_realizable_value_dict,
    bank_balance = bank_values,

    ft_value = 1.23,
    bench_factor = 1e-4,

    objective_column = 'custom_score',
    captain_column = 'captain_idx',

    # WC_WEEK = 24,
    # FH_WEEK = 34,
    # BB_WEEK = 33,
    # TC_WEEK = 26,

    fixed_player_dict = {
        'Default':[],
    },
    banned_player_dict = {
        'Default':[183, 221, 367],
    },
)